## Step‑by‑Step Implementation
### Step 1 — Install Required Libraries

Run this once at the top of your Colab notebook:

In [1]:
!pip install langchain langchain-community langchain-text-splitters \
sentence-transformers faiss-cpu transformers accelerate pypdf streamlit reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

- langchain → framework for document loaders, text splitters, and retrieval.

- sentence-transformers → embeddings model.

- faiss-cpu → vector database for similarity search.

- transformers → Hugging Face models (FLAN‑T5).

- pypdf → PDF parsing.

- reportlab → optional, for generating sample PDFs.

- streamlit → frontend (later, for deployment).


### Step 2 — Create or Upload PDFs

You can either upload your own PDFs:
```
from google.colab import files
uploaded = files.upload()
```
Or generate sample PDFs programmatically:

```
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

def create_pdf(filename, text):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [Paragraph(text, styles["Normal"])]
    doc.build(story)

create_pdf("ai.pdf", "Artificial Intelligence is used in healthcare, finance, and education.")
create_pdf("cybersecurity.pdf", "Cybersecurity protects systems from hackers. AI helps detect threats faster.")
create_pdf("data_science.pdf", "Data Science combines statistics, programming, and visualization for decision making.")

print("PDFs created!")
```

In [2]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

def create_pdf(filename, text):
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [Paragraph(text, styles["Normal"])]
    doc.build(story)

create_pdf("ai.pdf", "Artificial Intelligence is used in healthcare, finance, and education.")
create_pdf("cybersecurity.pdf", "Cybersecurity protects systems from hackers. AI helps detect threats faster.")
create_pdf("data_science.pdf", "Data Science combines statistics, programming, and visualization for decision making.")

print("PDFs created!")

PDFs created!


### Step 3 — Load Multiple PDFs
Use LangChain’s PyPDFLoader:

In [3]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdfs(files):
    docs = []
    for file in files:
        loader = PyPDFLoader(file)
        docs.extend(loader.load())
    return docs

files = ["ai.pdf", "cybersecurity.pdf", "data_science.pdf"]
documents = load_pdfs(files)
print("Pages loaded:", len(documents))

Pages loaded: 3


### Step 4 — Split into Chunks
Large documents must be split for embeddings:

In [6]:
# First, install the new package if not already:

!pip install langchain-text-splitters

In [9]:
from langchain_text_splitters import CharacterTextSplitter

def split_docs(documents):
    splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    return splitter.split_documents(documents)

chunks = split_docs(documents)
print("Chunks created:", len(chunks))

Chunks created: 3


## Step 5 — Create Embeddings + FAISS Vector DB

In [14]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert chunks to embeddings
doc_embeddings = embedder.encode([c.page_content for c in chunks])

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print("Vector DB ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB ready


## Step 6 — Load Free LLM (FLAN‑T5)

In [12]:
from transformers import pipeline

llm = pipeline(
    "text-generation",   # use text-generation in Transformers v5
    model="google/flan-t5-base",
    max_length=256
)

print("LLM Loaded ")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCa

LLM Loaded 🚀


## Step 7 — Retrieval + Chat Function

In [15]:
def ask_pdf(query, top_k=3, history=[]):
    # Embed query
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    # Build context from top chunks
    context = " ".join([chunks[i].page_content for i in indices[0]])

    # Build prompt with chat memory
    past = "".join([f"User: {h[0]} Bot: {h[1]}\n" for h in history])
    prompt = f"Conversation:\n{past}User: {query}\nContext: {context}\nBot:"

    # Generate answer
    output = llm(prompt, temperature=0.7)
    reply = output[0]["generated_text"]
    return reply

## Step 8 — Test the Chat

In [16]:
history = []
q = "What is the role of AI in healthcare?"
answer = ask_pdf(q, history=history)
history.append([q, answer])
print("Bot:", answer)

Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Bot: Conversation:
User: What is the role of AI in healthcare?
Context: Artificial Intelligence is used in healthcare, finance, and education. Cybersecurity protects systems from hackers. AI helps detect threats faster. Data Science combines statistics, programming, and visualization for decision making.
Bot:


## Key Concepts Explained

- Multi‑PDF support → load multiple files into one vector DB.

- Chat memory → history keeps track of past Q&A.

- FAISS → stores embeddings for fast similarity search.

- FLAN‑T5 → free Hugging Face model simulating an API‑style LLM.

- No API key → everything runs locally in Colab, avoiding paid APIs.